# 00 — Setup & Base Model

This is stage **0** of a from-scratch LLM post-training pipeline. It loads a small base model, inspects it, generates from it, and attaches a **hand-written LoRA** layer. Every later notebook (`01`–`04`) starts from this same foundation and loads the previous stage's LoRA checkpoint.

**Runs anywhere.** The code auto-selects CUDA → Apple Metal (MPS) → CPU. The base model is only ~135M parameters, so a MacBook with Metal is perfectly fine; a RunPod GPU just makes it faster (and enables bf16 mixed precision automatically).

Everything algorithmic lives in `src/` (`lora.py`, `data.py`, `losses.py`, `utils.py`) — the notebooks orchestrate and explain it.

## The post-training pipeline

A pretrained base model predicts the next token well, but it doesn't *know a
specific domain*, doesn't *follow instructions*, isn't *aligned to preferences*,
and can't *reason toward a checkable goal*. Each post-training stage fixes one of
these:

| Stage | Problem it solves | Data | Loss / objective |
|-------|-------------------|------|------------------|
| **CPT**  | Inject new domain knowledge / vocabulary | raw text | next-token loss on **all** tokens |
| **SFT**  | Make it follow instructions & format output | (prompt, response) | next-token loss **masked to the response** |
| **DPO**  | Align to human preferences (cheaply) | (prompt, chosen, rejected) | logistic loss vs a **frozen reference**, no reward model |
| **GRPO** | Improve reasoning toward a **verifiable** reward | prompts + reward fn | clipped policy gradient with a **group-relative baseline** |

```
base  ──▶  01 CPT  ──▶  02 SFT  ──▶  03 DPO  ──▶  04 GRPO
            (know)      (obey)       (align)      (reason)
```

Two contrasts to keep in mind:
- **CPT vs SFT** use the *same* loss; only the label masking differs.
- **DPO** removes the reward model of classic RLHF; **GRPO** removes the value
  network (critic) of PPO by using the group mean as the baseline.

In [1]:
import sys, pathlib

# Make the repo root importable so `import src...` works from notebooks/.
REPO_ROOT = pathlib.Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.utils import set_seed, get_device, autocast_dtype
from src.lora import inject_lora, count_parameters, LoRALinear

set_seed(42)
device = get_device()
print(f"Device: {device}")
print(f"Mixed-precision dtype (CUDA only): {autocast_dtype(device)}")

Device: mps
Mixed-precision dtype (CUDA only): torch.float32


/Users/jorgesandoval/code/llm-post-training-from-scratch/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## 1. Load the base model and tokenizer

This stage uses [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M):
small enough to iterate in seconds, real enough to show genuine learning. Using
`transformers` to load the model is fine — *"from scratch" refers to the training
algorithms*, not to re-implementing the transformer.

The pad token is set to the EOS token (SmolLM2 has no dedicated pad token) so
variable-length sequences can be batched later.

In [2]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.to(device)
model.eval()

print(model.config.architectures, "| hidden:", model.config.hidden_size,
      "| layers:", model.config.num_hidden_layers,
      "| vocab:", model.config.vocab_size)

['LlamaForCausalLM'] | hidden: 576 | layers: 30 | vocab: 49152


## 2. Parameter count

Before touching LoRA, here is the full size of the model. All of these
parameters are currently trainable — the next steps **freeze them** and train
only a tiny low-rank adapter.

In [3]:
trainable, total = count_parameters(model)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}  ({100 * trainable / total:.2f}%)")

Total parameters:     134,515,008
Trainable parameters: 134,515,008  (100.00%)


## 3. Baseline generation

Defines a small `generate` helper (reused in later notebooks) and shows what
the **untrained** base model says. Note especially how it responds to the
fictional domain — it has never heard of "lumenwrights" — and how it handles an
instruction. These baselines are what CPT and SFT will visibly change.

In [4]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=0.0):
    """Greedy (temperature=0) or sampled generation for quick qualitative checks."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=0.95 if temperature > 0 else None,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()


print("PROMPT: A lumenwright is")
print("->", generate(model, "A lumenwright is", max_new_tokens=40))
print("\nPROMPT: ### Instruction:\\nWhat is a lumenwright?\\n\\n### Response:\\n")
print("->", generate(model, "### Instruction:\nWhat is a lumenwright?\n\n### Response:\n", max_new_tokens=40))

PROMPT: A lumenwright is


-> a person who designs and builds lumen windows.

Lumen windows are designed to be used in a variety of applications, including:

  • Windows for the interior of a building
  • Windows

PROMPT: ### Instruction:\nWhat is a lumenwright?\n\n### Response:\n


-> A lumenwright is a person who designs and builds lumen windows.

### Instruction:

A lumenwright is a person who designs and builds lumen windows.


## 4. Inject the hand-written LoRA

Instead of updating all 135M weights, they are frozen and a small low-rank
update is learned on the attention **query** and **value** projections. The
`LoRALinear` (see `src/lora.py`) replaces an `nn.Linear` with:

\[ y = \underbrace{xW^\top + b}_{\text{frozen}} + \frac{\alpha}{r}\,(xA^\top)B^\top \]

where \(A \in \mathbb{R}^{r\times d_{in}}\), \(B \in \mathbb{R}^{d_{out}\times r}\),
and **B is initialized to zero** so the adapted model is *identical* to the
original at step 0. Only `A` and `B` are trained.

`inject_lora` first freezes every parameter, then swaps each matching `q_proj` /
`v_proj` for a `LoRALinear`. The result: a few hundred-thousand trainable params
instead of 135M.

In [5]:
n_wrapped = inject_lora(model, target_modules=("q_proj", "v_proj"), r=8, alpha=16)
model.to(device)  # newly created LoRA params need to be on the same device

trainable, total = count_parameters(model)
print(f"LoRALinear layers injected: {n_wrapped}")
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}  ({100 * trainable / total:.3f}%)")

# Peek at one injected layer.
example = next(m for m in model.modules() if isinstance(m, LoRALinear))
print("\nExample injected layer:\n ", example)

LoRALinear layers injected: 60
Total parameters:     134,975,808
Trainable parameters: 460,800  (0.341%)

Example injected layer:
  LoRALinear(
  in_features=576, out_features=576, r=8, alpha=16, scaling=2.0000
  (base): Linear(in_features=576, out_features=576, bias=False)
  (lora_dropout): Identity()
)


### Sanity check: LoRA starts as a no-op

Because `B = 0`, the LoRA-wrapped model must produce **exactly** the same logits
as the frozen base model before any training. The check below confirms the
maximum difference is ~0 (up to floating point).

In [6]:
with torch.no_grad():
    ids = tokenizer("The quick brown fox", return_tensors="pt").to(device)
    logits_lora = model(**ids).logits

    # Temporarily perturb B to show the adapter *can* change the output.
    layer = next(m for m in model.modules() if isinstance(m, LoRALinear))
    layer.lora_B.add_(0.01)
    logits_perturbed = model(**ids).logits
    layer.lora_B.sub_(0.01)  # restore

print(f"max |logits_lora - logits_base| at init : {(logits_lora - model(**ids).logits).abs().max().item():.2e}")
print(f"max change after nudging one B          : {(logits_perturbed - logits_lora).abs().max().item():.2e}")

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


max |logits_lora - logits_base| at init : 0.00e+00
max change after nudging one B          : 6.39e-03


## Recap & what's next

- Loaded `SmolLM2-135M`, confirmed it has no knowledge of the fictional domain.
- Froze all weights and injected a **manual LoRA** into `q_proj`/`v_proj` — now
  only ~0.x% of parameters are trainable.
- Verified the adapter is a no-op at initialization (B = 0).

➡️ **`01_cpt.ipynb`** — Continued Pre-Training: trains this LoRA with a plain
next-token loss over the raw domain corpus, watching the model *absorb the
vocabulary*. The adapter is saved to `checkpoints/cpt/` for the next stage.